In [ ]:
"""
QC: Count 'bad' windows ONLY for keypoints used by saved metrics.

Definitions
-----------
- Keypoint present in a frame if prob_i >= CONFIDENCE_THRESHOLD and x_i,y_i are non-NaN.
- In a window, a keypoint is 'bad' if the longest consecutive run of missing frames > MAX_INTERP.
- A metric is 'bad' in a window if ANY of its constituent keypoints is bad in that window.

Outputs
-------
1) keypoint_bad_windows.csv: per-file, per-keypoint bad window counts and %.
2) metric_bad_windows.csv:   per-file, per-metric bad window counts and %.

Adjust
------
- INPUT_DIR, OUTPUT_DIR
- WINDOW_SIZE, OVERLAP, CONFIDENCE_THRESHOLD, MAX_INTERP
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# --------- Config ---------
OUTPUT_DIR = "qc_outputs"
WINDOW_SIZE = 1800
OVERLAP = 0.0                 # 0=no overlap; 0.5=50% overlap
CONFIDENCE_THRESHOLD = 0.3
MAX_INTERP = 60
# --------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Keypoints actually used by saved metrics
METRIC_KPS = {
    "eyes":       [37, 38, 40, 41, 43, 44, 46, 47],
    "head_rotation":    [36, 45],
    "mouth_dist":       [62, 66],
    "pupils_combined":  [68, 69],
}
RELEVANT_KPS = sorted({kp for kps in METRIC_KPS.values() for kp in kps})

def load_csv(fp: str) -> pd.DataFrame:
    return pd.read_csv(fp)

def window_ranges(n_rows: int, window_size: int, overlap: float):
    if n_rows < window_size:
        return []
    step = max(1, int(round(window_size * (1 - overlap))))
    return [(s, s + window_size) for s in range(0, n_rows - window_size + 1, step)]

def _max_true_run_length(b: pd.Series) -> int:
    run = max_run = 0
    arr = b.to_numpy()
    for v in arr:
        if v:
            run += 1
            if run > max_run:
                max_run = run
        else:
            run = 0
    return max_run

def kp_missing_series(df_w: pd.DataFrame, i: int) -> pd.Series:
    """True when keypoint i is missing in a frame (prob<thresh OR x/y NaN)."""
    xcol, ycol, pcol = f"x{i}", f"y{i}", f"prob{i}"
    if (xcol not in df_w.columns) or (ycol not in df_w.columns) or (pcol not in df_w.columns):
        return pd.Series(True, index=df_w.index)  # treat as fully missing if absent
    good = (df_w[pcol] >= CONFIDENCE_THRESHOLD) & df_w[xcol].notna() & df_w[ycol].notna()
    return ~good

def analyze_file(fp: str):
    df = load_csv(fp)
    base = os.path.basename(fp)
    wranges = window_ranges(len(df), WINDOW_SIZE, OVERLAP)
    total_windows = len(wranges)

    # early out if too short for one window
    if total_windows == 0:
        kp_rows = [{
            "file": base, "keypoint": i, "bad_windows": 0,
            "total_windows": 0, "pct_bad": np.nan
        } for i in RELEVANT_KPS]
        met_rows = [{
            "file": base, "metric": m, "bad_windows": 0,
            "total_windows": 0, "pct_bad": np.nan
        } for m in METRIC_KPS.keys()]
        return pd.DataFrame(kp_rows), pd.DataFrame(met_rows)

    # Precompute per-window per-kp "is_bad_in_window" flags
    kp_bad_counts = {i: 0 for i in RELEVANT_KPS}
    metric_bad_counts = {m: 0 for m in METRIC_KPS}

    for (s, e) in wranges:
        df_w = df.iloc[s:e].reset_index(drop=True)

        # keypoint-level
        kp_bad_in_this_window = {}
        for i in RELEVANT_KPS:
            missing = kp_missing_series(df_w, i)
            L = _max_true_run_length(missing)
            is_bad = (L > MAX_INTERP)
            kp_bad_in_this_window[i] = is_bad
            if is_bad:
                kp_bad_counts[i] += 1

        # metric-level: bad if ANY of its kps is bad in this window
        for m, kps in METRIC_KPS.items():
            if any(kp_bad_in_this_window.get(i, True) for i in kps):
                metric_bad_counts[m] += 1

    # Build outputs
    kp_rows = []
    for i in RELEVANT_KPS:
        bw = kp_bad_counts[i]
        kp_rows.append({
            "file": base,
            "keypoint": i,
            "bad_windows": bw,
            "total_windows": total_windows,
            "pct_bad": bw / total_windows if total_windows > 0 else np.nan
        })

    met_rows = []
    for m in METRIC_KPS:
        bw = metric_bad_counts[m]
        met_rows.append({
            "file": base,
            "metric": m,
            "bad_windows": bw,
            "total_windows": total_windows,
            "pct_bad": bw / total_windows if total_windows > 0 else np.nan
        })

    return pd.DataFrame(kp_rows), pd.DataFrame(met_rows)

def main():
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]
    all_kp = []
    all_met = []
    for f in tqdm(files, desc="QC scanning"):
        fp = os.path.join(INPUT_DIR, f)
        try:
            kp_df, met_df = analyze_file(fp)
        except Exception as e:
            kp_df = pd.DataFrame([{
                "file": f, "keypoint": None, "bad_windows": None,
                "total_windows": None, "pct_bad": None, "error": str(e)
            }])
            met_df = pd.DataFrame([{
                "file": f, "metric": None, "bad_windows": None,
                "total_windows": None, "pct_bad": None, "error": str(e)
            }])
        all_kp.append(kp_df)
        all_met.append(met_df)

    out_kp = pd.concat(all_kp, ignore_index=True)
    out_met = pd.concat(all_met, ignore_index=True)

    out_kp_path = os.path.join(OUTPUT_DIR, "keypoint_bad_windows.csv")
    out_met_path = os.path.join(OUTPUT_DIR, "metric_bad_windows.csv")
    out_kp.to_csv(out_kp_path, index=False)
    out_met.to_csv(out_met_path, index=False)
    print(f"Wrote:\n  {out_kp_path}\n  {out_met_path}")

if __name__ == "__main__":
    main()


QC scanning: 100%|██████████| 216/216 [00:47<00:00,  4.50it/s]

Wrote:
  qc_outputs/keypoint_bad_windows.csv
  qc_outputs/metric_bad_windows.csv
